# Gestión de Duplicados: Identificación con `.duplicated()`

## 🎯 Objetivos
- Comprender cómo identificar filas duplicadas en un DataFrame.
- Dominar el parámetro `keep` para controlar qué ocurrencias se marcan como duplicadas.
- Aplicar la detección de duplicados para encontrar valores extremos (mínimos/máximos) por categoría.

## 📖 Introducción

Los datos duplicados son uno de los problemas más comunes en la limpieza de datos. Pueden surgir por errores en la recolección, uniones (joins) mal ejecutadas o entradas redundantes. Dejar duplicados en un análisis puede sesgar las estadísticas (como el promedio o la suma) y llevar a conclusiones erróneas.

El método `.duplicated()` de Pandas no elimina los datos, sino que genera una **Máscara Booleana** que nos indica exactamente qué filas son repeticiones de otras.

## 🌉 Puente Pedagógico: El Concepto de "El Primer Encuentro"

Cuando Pandas analiza una columna buscando duplicados, actúa como un bibliotecario que anota los libros que ya ha visto. 

Por defecto, la primera vez que encuentra un valor, dice: "Este es nuevo" (`False` para duplicado). La segunda, tercera y siguiente vez que ve el mismo valor, dice: "Este ya lo tengo" (`True` para duplicado).

### Visualización de la Lógica
```
[ Fila de Datos ]  -->  [ .duplicated() ]  -->  [ Máscara Booleana ]
  Laptop A (1ra)  ---------------------------->  False (Único)
  Laptop A (2da)  ---------------------------->  True  (Duplicado)
  Laptop A (3ra)  ---------------------------->  True  (Duplicado)
  Laptop B (1ra)  ---------------------------->  False (Único)
```

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('laptop_price.csv')
df_laptops = pd.read_csv(DATA_PATH)
df_laptops.head()

## 🛠️ Implementación

### 1. Detección en una sola columna

Podemos verificar si hay IDs de laptop repetidos. En un dataset limpio, el ID debería ser único.

In [ ]:
# Crear máscara de duplicados para la columna 'laptop_ID'
mask_id_dup = df_laptops.duplicated('laptop_ID')

# Contar cuántos duplicados existen
num_dups = mask_id_dup.sum()
print(f"Se encontraron {num_dups} IDs duplicados.")

# Mostrar las filas que son duplicadas
df_laptops[mask_id_dup]

### 2. Detección en múltiples columnas

A veces, una fila no es duplicada en todas sus columnas, pero sí en un conjunto específico (ej. mismo producto, mismo tipo y misma pantalla).

In [ ]:
# Definir el conjunto de columnas a analizar
cols_to_check = ['Product', 'TypeName', 'Inches']

# Detectar duplicados basados en ese conjunto
mask_multi_dup = df_laptops.duplicated(subset=cols_to_check)

df_laptops[mask_multi_dup].sort_values('Product').head()

### 3. El poder del parámetro `keep`

Podemos cambiar el comportamiento de `.duplicated()` usando el argumento `keep`:

- `keep='first'` (Default): Marca como `True` todas las ocurrencias EXCEPTO la primera.
- `keep='last'`: Marca como `True` todas las ocurrencias EXCEPTO la última.
- `keep=False`: Marca TODAS las ocurrencias de un valor duplicado como `True`.

In [ ]:
# Comparación de 'keep'
dup_first = df_laptops.duplicated('Company', keep='first')
dup_last = df_laptops.duplicated('Company', keep='last')
dup_all = df_laptops.duplicated('Company', keep=False)

print(f"Duplicados (first): {dup_first.sum()}")
print(f"Duplicados (last): {dup_last.sum()}")
print(f"Duplicados (all): {dup_all.sum()}")

### 4. Aplicación Real: Encontrar Extremos por Categoría

Si combinamos `sort_values()` con `.duplicated()`, podemos encontrar fácilmente el elemento más barato o más caro de cada marca sin usar `groupby()`.

In [ ]:
# 1. Ordenar por Marca (asc) y Precio (asc)
df_sorted = df_laptops.sort_values(['Company', 'Price_euros'])

# 2. La primera ocurrencia de cada marca será la más barata
cheapest_laptops = df_sorted[~df_sorted.duplicated('Company', keep='first')]

# 3. La última ocurrencia de cada marca será la más cara
most_expensive_laptops = df_sorted[~df_sorted.duplicated('Company', keep='last')]

print("--- Laptops más baratas por marca ---")
print(cheapest_laptops[['Company', 'Product', 'Price_euros']].head())

print("\n--- Laptops más caras por marca ---")
print(most_expensive_laptops[['Company', 'Product', 'Price_euros']].head())

## 📝 Ejercicios de Práctica

1. **Limpieza Total**: Identifica cuántas filas son exactamente idénticas en todas sus columnas.
2. **Análisis de Hardware**: Encuentra todas las laptops que comparten el mismo `Cpu` y `Ram`, pero que tienen diferentes precios (pista: usa `keep=False` para ver todas las instancias).
3. **Unicidad**: Crea un nuevo DataFrame que contenga solo una instancia única de cada `Product`, conservando la que tenga el precio más alto.

In [ ]:
# Ejercicio 1

# Ejercicio 2

# Ejercicio 3


## 📋 Resumen Rápido

| Valor de `keep` | Resultado |
| :--- | :--- | 
| `'first'` | Marca como duplicado todo lo que venga después de la primera aparición. |
| `'last'` | Marca como duplicado todo lo que venga antes de la última aparición. |
| `False` | Marca absolutamente todas las filas que tengan un valor repetido en cualquier lugar. |